# ML-08 - Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshenDary/Week1_RunTheStarterNotebooks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook trains one simple model for Lane 2 - Refresh / Content Opportunity Scoring - and compares it against the Week-4 ranked-rule baseline on the same data and same Precision@K metrics.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

I am using **one depth-limited Decision Tree** for the Week-5 model.

This fits Lane 2 because the actual decision is a ranked editor queue: which content should be reviewed first for a possible title/meta/content refresh? The Week-4 baseline is already a transparent threshold rule around visible pages, low CTR versus expected CTR, impression volume, and staleness. A small tree is the closest honest next step: it can learn threshold interactions among those same signals, produce a probability-like score for ranking, and still be printed and read.

I am not using Random Forest or Gradient Boosting here. The starter data has 30,000 rows, but the target is still a proxy derived from search trend direction, and the Week-4 baseline is simple. A more complex ensemble would be harder to explain to an editor and would reward complexity before a small tree earns it.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier, export_text

RANDOM_STATE = 42
N_SPLITS = 5
TREE_MAX_DEPTH = 4
MIN_SAMPLES_LEAF = 100

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")


def expected_ctr_by_position(avg_position: pd.Series) -> pd.Series:
    return pd.Series(
        np.select(
            [
                avg_position.between(0.01, 3, inclusive="both"),
                avg_position.between(3.01, 10, inclusive="both"),
                avg_position.between(10.01, 20, inclusive="both"),
            ],
            [2.00, 1.00, 0.50],
            default=np.nan,
        ),
        index=avg_position.index,
    )


def build_model_frame(raw_frame: pd.DataFrame) -> pd.DataFrame:
    frame = raw_frame.copy()
    frame["is_declining_label"] = frame["trend_direction"].eq("down").astype(int)
    frame["expected_ctr"] = expected_ctr_by_position(frame["avg_position"])
    frame["visible_valid_position"] = (
        frame["impressions_90d"].ge(300)
        & frame["avg_position"].gt(0)
        & frame["avg_position"].le(20)
    ).astype(int)
    frame["ctr_gap_score"] = (
        (frame["expected_ctr"] - frame["ctr"]) / frame["expected_ctr"]
    ).clip(lower=0, upper=1).fillna(0.0)
    frame["log_impressions_90d"] = np.log1p(frame["impressions_90d"])
    frame["freshness_score"] = (frame["days_since_last_update"] / 180).clip(lower=0, upper=1)
    return frame


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"
BASELINE_PATH = REPO_ROOT / "work/outputs/baseline_action_score.csv"
BASELINE_METRICS_PATH = REPO_ROOT / "work/outputs/baseline_action_score_metrics.json"

raw_df = pd.read_csv(DATA_PATH)
baseline_df = pd.read_csv(BASELINE_PATH)
baseline_receipt = json.loads(BASELINE_METRICS_PATH.read_text())
model_df = build_model_frame(raw_df)

feature_columns = [
    "ctr",
    "avg_position",
    "visible_valid_position",
    "ctr_gap_score",
    "log_impressions_90d",
    "days_since_last_update",
    "freshness_score",
]
forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_id",
    "client_id",
}

assert set(feature_columns).isdisjoint(forbidden_inputs), "Model feature list contains leakage, trend-window, or ID columns."
assert len(model_df) == baseline_receipt["rows"] == len(baseline_df), "Baseline and model data row counts do not match."

method_receipt = pd.DataFrame(
    {
        "item": [
            "rows",
            "clients",
            "positive label rate",
            "model method",
            "random seed",
            "tree max_depth",
            "min_samples_leaf",
            "feature columns",
            "forbidden feature overlap",
        ],
        "value": [
            f"{len(model_df):,}",
            model_df["client_id"].nunique(),
            f"{model_df['is_declining_label'].mean():.1%}",
            "DecisionTreeClassifier",
            RANDOM_STATE,
            TREE_MAX_DEPTH,
            MIN_SAMPLES_LEAF,
            ", ".join(feature_columns),
            sorted(set(feature_columns).intersection(forbidden_inputs)),
        ],
    }
)
display(method_receipt)


,item,value
0,rows,"30,000"
1,clients,32
2,positive label rate,54.2%
3,model method,DecisionTreeClassifier
4,random seed,42
5,tree max_depth,4
6,min_samples_leaf,100
7,feature columns,"ctr, avg_position, visible_valid_position, ctr..."
8,forbidden feature overlap,[]


## 2. Split design

I checked the Week-4 baseline notebook and artifact before modeling. The baseline did **not** use a train/test split; it ranked all 30,000 rows and reported Precision@10 and Precision@50 against `is_declining_label` on that full queue.

For a learned model, using the same rows for both training and scoring would be too generous. I keep the same full-queue evaluation surface as Week 4, but each model score below is **out-of-fold** from a 5-fold split grouped by `client_id`. That means every row receives a score from a tree that did not train on that row's client. A time split is not available in this starter CSV because each row is a trailing-90-day snapshot, not a dated panel; the later warehouse contract handles time-aware windows separately.


In [2]:
y = model_df["is_declining_label"].astype(int)
groups = model_df["client_id"].fillna("unknown").astype(str)
fold_assignments = np.zeros(len(model_df), dtype=int)
fold_rows: list[dict[str, object]] = []

group_kfold = GroupKFold(n_splits=N_SPLITS)
for fold_number, (train_idx, test_idx) in enumerate(group_kfold.split(model_df[feature_columns], y, groups), start=1):
    train_clients = set(groups.iloc[train_idx])
    test_clients = set(groups.iloc[test_idx])
    overlap = train_clients.intersection(test_clients)
    assert not overlap, "A client appears in both train and test for a fold."
    fold_assignments[test_idx] = fold_number
    fold_rows.append(
        {
            "fold": fold_number,
            "train_rows": len(train_idx),
            "test_rows": len(test_idx),
            "train_clients": len(train_clients),
            "test_clients": len(test_clients),
            "test_positive_rate": y.iloc[test_idx].mean(),
        }
    )

fold_table = pd.DataFrame(fold_rows)
fold_table["test_positive_rate"] = fold_table["test_positive_rate"].map(lambda value: f"{value:.1%}")

print("Week-4 baseline split found: no train/test split; full ranked queue evaluation.")
print("Week-5 model scoring split: GroupKFold by client_id for out-of-fold predictions.")
display(fold_table)
assert (fold_assignments > 0).all(), "Every row should be assigned to one held-out fold."


Week-4 baseline split found: no train/test split; full ranked queue evaluation.
Week-5 model scoring split: GroupKFold by client_id for out-of-fold predictions.


,fold,train_rows,test_rows,train_clients,test_clients,test_positive_rate
0,1,22992,7008,31,1,49.0%
1,2,24269,5731,25,7,64.5%
2,3,24247,5753,24,8,37.9%
3,4,24245,5755,24,8,62.2%
4,5,24247,5753,24,8,58.5%


## 3. Train + compare vs my baseline

The comparison below uses the same rows and the same Week-4 metrics: Precision@10 and Precision@50, with the full-data base rate shown for context. The baseline score is the Week-4 artifact. The model score is the out-of-fold probability from the small decision tree.

Because tree leaves can create ties, both rankings use the same deterministic tie-breakers after the main score: higher `impressions_90d`, then higher `ctr_gap_score`, then `content_id` only as a stable sorter. `content_id` is not a feature.


In [3]:
def clean_feature_frame(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    features = frame[columns].apply(pd.to_numeric, errors="coerce")
    features = features.replace([np.inf, -np.inf], np.nan)
    return features.fillna(features.median(numeric_only=True))


def train_tree() -> DecisionTreeClassifier:
    return DecisionTreeClassifier(
        max_depth=TREE_MAX_DEPTH,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


def ranked_by_score(frame: pd.DataFrame, score_col: str) -> pd.DataFrame:
    return frame.sort_values(
        [score_col, "impressions_90d", "ctr_gap_score", "content_id"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)


def precision_at_k_from_ranked(ranked_frame: pd.DataFrame, k: int) -> float:
    return float(ranked_frame.head(k)["is_declining_label"].mean())


X = clean_feature_frame(model_df, feature_columns)
out_of_fold_probability = np.zeros(len(model_df), dtype=float)
fold_metric_rows: list[dict[str, object]] = []

for fold_number, (train_idx, test_idx) in enumerate(group_kfold.split(X, y, groups), start=1):
    tree = train_tree()
    tree.fit(X.iloc[train_idx], y.iloc[train_idx])
    fold_scores = tree.predict_proba(X.iloc[test_idx])[:, 1]
    out_of_fold_probability[test_idx] = fold_scores
    fold_ranked = ranked_by_score(
        model_df.iloc[test_idx].assign(model_probability=fold_scores),
        "model_probability",
    )
    fold_metric_rows.append(
        {
            "fold": fold_number,
            "test_rows": len(test_idx),
            "test_base_rate": y.iloc[test_idx].mean(),
            "model_precision_at_10_in_fold": precision_at_k_from_ranked(fold_ranked, 10),
            "model_precision_at_50_in_fold": precision_at_k_from_ranked(fold_ranked, 50),
        }
    )

assert np.isfinite(out_of_fold_probability).all(), "Model produced a non-finite score."
model_df["model_probability"] = out_of_fold_probability
model_df["model_fold"] = fold_assignments

baseline_lookup = baseline_df.set_index("content_id")["score"]
model_df["baseline_score"] = model_df["content_id"].map(baseline_lookup)
assert model_df["baseline_score"].notna().all(), "Could not align every row to the Week-4 baseline score."

baseline_ranked = ranked_by_score(model_df, "baseline_score")
model_ranked = ranked_by_score(model_df, "model_probability")

comparison_table = pd.DataFrame(
    [
        {
            "method": "Week-4 hand rule baseline",
            "scoring": "full queue rule score",
            "rows_scored": len(model_df),
            "base_rate": y.mean(),
            "precision_at_10": precision_at_k_from_ranked(baseline_ranked, 10),
            "precision_at_50": precision_at_k_from_ranked(baseline_ranked, 50),
            "roc_auc_diagnostic": roc_auc_score(y, model_df["baseline_score"]),
            "average_precision_diagnostic": average_precision_score(y, model_df["baseline_score"]),
        },
        {
            "method": "Week-5 depth-4 decision tree",
            "scoring": "client-grouped out-of-fold probability",
            "rows_scored": len(model_df),
            "base_rate": y.mean(),
            "precision_at_10": precision_at_k_from_ranked(model_ranked, 10),
            "precision_at_50": precision_at_k_from_ranked(model_ranked, 50),
            "roc_auc_diagnostic": roc_auc_score(y, model_df["model_probability"]),
            "average_precision_diagnostic": average_precision_score(y, model_df["model_probability"]),
        },
    ]
)

for metric_col in ["base_rate", "precision_at_10", "precision_at_50", "roc_auc_diagnostic", "average_precision_diagnostic"]:
    comparison_table[metric_col] = comparison_table[metric_col].round(4)

fold_metric_table = pd.DataFrame(fold_metric_rows)
for metric_col in ["test_base_rate", "model_precision_at_10_in_fold", "model_precision_at_50_in_fold"]:
    fold_metric_table[metric_col] = fold_metric_table[metric_col].round(4)

print("Fold-level model checks, before the final full-queue table:")
display(fold_metric_table)

print("Model-vs-baseline table: same rows, same Week-4 Precision@K metrics.")
display(comparison_table)

assert round(comparison_table.loc[0, "precision_at_10"], 4) == baseline_receipt["precision_at_10"]
assert round(comparison_table.loc[0, "precision_at_50"], 4) == baseline_receipt["precision_at_50"]

final_tree = train_tree()
final_tree.fit(X, y)
importance_table = (
    pd.DataFrame({"feature": feature_columns, "importance": final_tree.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print("Feature importance from a final tree fitted on all rows for interpretation only, not for headline scoring:")
display(importance_table)

print("Readable final tree, fitted on all rows for interpretation only:")
print(export_text(final_tree, feature_names=feature_columns, decimals=3))


Fold-level model checks, before the final full-queue table:


,fold,test_rows,test_base_rate,model_precision_at_10_in_fold,model_precision_at_50_in_fold
0,1,7008,0.4902,0.4,0.42
1,2,5731,0.6454,0.8,0.72
2,3,5753,0.3795,0.9,0.70
3,4,5755,0.6222,0.5,0.76
4,5,5753,0.5847,0.4,0.60


Model-vs-baseline table: same rows, same Week-4 Precision@K metrics.


,method,scoring,rows_scored,base_rate,precision_at_10,precision_at_50,roc_auc_diagnostic,average_precision_diagnostic
0,Week-4 hand rule baseline,full queue rule score,30000,0.5421,0.6,0.62,0.5690,0.5945
1,Week-5 depth-4 decision tree,client-grouped out-of-fold probability,30000,0.5421,0.5,0.76,0.6145,0.6147


Feature importance from a final tree fitted on all rows for interpretation only, not for headline scoring:


,feature,importance
0,log_impressions_90d,0.730027
1,avg_position,0.112137
2,ctr_gap_score,0.108831
3,freshness_score,0.029551
4,days_since_last_update,0.019454
5,ctr,0.000000
6,visible_valid_position,0.000000


Readable final tree, fitted on all rows for interpretation only:
|--- log_impressions_90d <= 1.869
|   |--- avg_position <= 0.750
|   |   |--- log_impressions_90d <= 0.896
|   |   |   |--- class: 0
|   |   |--- log_impressions_90d >  0.896
|   |   |   |--- days_since_last_update <= 7.000
|   |   |   |   |--- class: 0
|   |   |   |--- days_since_last_update >  7.000
|   |   |   |   |--- class: 0
|   |--- avg_position >  0.750
|   |   |--- log_impressions_90d <= 1.242
|   |   |   |--- avg_position <= 4.750
|   |   |   |   |--- class: 0
|   |   |   |--- avg_position >  4.750
|   |   |   |   |--- class: 0
|   |   |--- log_impressions_90d >  1.242
|   |   |   |--- days_since_last_update <= 95.000
|   |   |   |   |--- class: 0
|   |   |   |--- days_since_last_update >  95.000
|   |   |   |   |--- class: 0
|--- log_impressions_90d >  1.869
|   |--- ctr_gap_score <= 0.845
|   |   |--- avg_position <= 44.250
|   |   |   |--- freshness_score <= 0.117
|   |   |   |   |--- class: 1
|   |   |   |--

## 4. Errors and interpretation

The model mostly learns a version of the same story as the Week-4 rule: pages with real visibility, weak CTR versus a broad position expectation, and enough impression volume rise to the top. The important error pattern is that the model can still recommend non-declining pages when they look like obvious CTR opportunities; that is a proxy-label problem, not necessarily a bad content recommendation.

The misses are the opposite shape: declining pages that do not look like visible low-CTR opportunities fall down the queue. For Lane 2, that means this model is better at finding one kind of refresh opportunity than at finding every possible cause of decline.


In [4]:
model_top50 = model_ranked.head(50).copy()
baseline_top50 = baseline_ranked.head(50).copy()

model_top50_counts = pd.DataFrame(
    {
        "bucket": ["true positives in top 50", "false positives in top 50"],
        "count": [
            int(model_top50["is_declining_label"].sum()),
            int((1 - model_top50["is_declining_label"]).sum()),
        ],
    }
)
display(model_top50_counts)

error_profile = (
    model_top50.assign(error_type=np.where(model_top50["is_declining_label"].eq(1), "true_positive", "false_positive"))
    .groupby("error_type")
    .agg(
        n=("is_declining_label", "size"),
        median_model_probability=("model_probability", "median"),
        median_impressions_90d=("impressions_90d", "median"),
        median_ctr=("ctr", "median"),
        median_avg_position=("avg_position", "median"),
        median_days_since_last_update=("days_since_last_update", "median"),
        median_ctr_gap_score=("ctr_gap_score", "median"),
    )
    .reset_index()
)
display(error_profile)

false_positive_examples = (
    model_top50.loc[model_top50["is_declining_label"].eq(0)]
    .head(5)
    .reset_index()
    .rename(columns={"index": "model_rank"})
    .loc[
        :,
        [
            "model_rank",
            "model_probability",
            "baseline_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update",
            "ctr_gap_score",
        ],
    ]
)
print("Concrete false positives near the top of the model queue, with IDs withheld:")
display(false_positive_examples)

missed_positive_examples = (
    model_ranked.loc[model_ranked["is_declining_label"].eq(1)]
    .tail(5)
    .reset_index()
    .rename(columns={"index": "model_rank"})
    .loc[
        :,
        [
            "model_rank",
            "model_probability",
            "baseline_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "visible_valid_position",
            "days_since_last_update",
            "ctr_gap_score",
        ],
    ]
)
print("Concrete false negatives near the bottom of the model queue, with IDs withheld:")
display(missed_positive_examples)

segment_summary = (
    model_df.assign(
        score_decile=pd.qcut(model_df["model_probability"], q=10, duplicates="drop"),
        visible_bucket=np.where(model_df["visible_valid_position"].eq(1), "visible valid position", "not visible/valid"),
    )
    .groupby("visible_bucket")
    .agg(
        rows=("is_declining_label", "size"),
        observed_decline_rate=("is_declining_label", "mean"),
        mean_model_probability=("model_probability", "mean"),
        median_baseline_score=("baseline_score", "median"),
    )
    .reset_index()
)
for col in ["observed_decline_rate", "mean_model_probability", "median_baseline_score"]:
    segment_summary[col] = segment_summary[col].round(4)

display(segment_summary)

model_p10 = comparison_table.loc[comparison_table["method"].eq("Week-5 depth-4 decision tree"), "precision_at_10"].iloc[0]
model_p50 = comparison_table.loc[comparison_table["method"].eq("Week-5 depth-4 decision tree"), "precision_at_50"].iloc[0]
baseline_p10 = comparison_table.loc[comparison_table["method"].eq("Week-4 hand rule baseline"), "precision_at_10"].iloc[0]
baseline_p50 = comparison_table.loc[comparison_table["method"].eq("Week-4 hand rule baseline"), "precision_at_50"].iloc[0]

print(
    f"Interpretation: the tree gets {model_p50:.0%} of the top 50 right versus {baseline_p50:.0%} for the baseline, "
    f"but only {model_p10:.0%} of the top 10 versus {baseline_p10:.0%} for the baseline."
)
print(
    "That is a mixed result: the model is useful for a batch review queue, but the hand rule is still stronger at the very top of the list."
)


,bucket,count
0,true positives in top 50,38
1,false positives in top 50,12


,error_type,n,median_model_probability,median_impressions_90d,median_ctr,median_avg_position,median_days_since_last_update,median_ctr_gap_score
0,false_positive,12,0.79569,5723.0,0.075,4.05,20.0,0.94
1,true_positive,38,0.79569,2784.0,0.110,3.75,33.0,0.91


Concrete false positives near the top of the model queue, with IDs withheld:


,model_rank,model_probability,baseline_score,impressions_90d,ctr,avg_position,days_since_last_update,ctr_gap_score
0,0,0.79569,85.8257,272144,0.03,2.3,20,0.985
1,1,0.79569,81.5107,89761,0.09,4.0,20,0.910
2,2,0.79569,86.1163,46866,0.03,4.6,40,0.970
3,4,0.79569,85.8657,42310,0.13,3.5,104,0.870
4,5,0.79569,89.1067,39852,0.07,5.0,104,0.930


Concrete false negatives near the bottom of the model queue, with IDs withheld:


,model_rank,model_probability,baseline_score,impressions_90d,ctr,avg_position,visible_valid_position,days_since_last_update,ctr_gap_score
0,29552,0.010384,0.0,4,25.0,0.5,0,20,0.0
1,29619,0.010384,0.0,1,0.0,0.0,0,92,0.0
2,29675,0.006966,0.0,3,0.0,0.0,0,104,0.0
3,29682,0.006966,0.0,2,0.0,0.0,0,20,0.0
4,29748,0.006966,0.0,1,0.0,0.0,0,20,0.0


,visible_bucket,rows,observed_decline_rate,mean_model_probability,median_baseline_score
0,not visible/valid,16795,0.491,0.4603,0.0000
1,visible valid position,13205,0.607,0.5782,66.4952


Interpretation: the tree gets 76% of the top 50 right versus 62% for the baseline, but only 50% of the top 10 versus 60% for the baseline.
That is a mixed result: the model is useful for a batch review queue, but the hand rule is still stronger at the very top of the list.


## 5. Self-check

- Does this beat the baseline? **Mixed.** It beats the Week-4 baseline on Precision@50, but it does not beat the baseline on Precision@10.
- Is the improvement worth the added complexity? **Only if the workflow is a batch of about 50 pages.** For a tiny top-10 editor queue, I would keep the Week-4 rule because it is simpler and did better at that cutoff.
- Did I use the same data and metric as Week 4? **Yes.** The comparison table uses the same 30,000-row queue and the same Precision@10 / Precision@50 metrics. The model scores are out-of-fold by client so the learned model does not get an in-sample advantage.
- Did I avoid leakage? **Yes.** The model excludes `trend_direction`, `trend_pct`, all last/previous 30-day trend-window columns, and both ID columns as features.
- Honest final claim: the small tree is a measured, directional improvement for the top-50 review batch, not proof that ML should replace the baseline everywhere.


In [5]:
assert set(feature_columns).isdisjoint(forbidden_inputs), "Leakage or ID column used as a model feature."
assert comparison_table.shape[0] == 2, "Final comparison should contain exactly baseline and one model."
assert comparison_table["rows_scored"].nunique() == 1, "Baseline and model must score the same number of rows."
assert round(comparison_table.loc[0, "precision_at_10"], 4) == baseline_receipt["precision_at_10"], "Baseline Precision@10 does not match Week-4 receipt."
assert round(comparison_table.loc[0, "precision_at_50"], 4) == baseline_receipt["precision_at_50"], "Baseline Precision@50 does not match Week-4 receipt."
assert np.isfinite(model_df["model_probability"]).all(), "Model scores must be finite."
assert model_df["model_fold"].between(1, N_SPLITS).all(), "Every row must have a held-out model fold."

beats_p10 = model_p10 > baseline_p10
beats_p50 = model_p50 > baseline_p50
worth_complexity = beats_p50 and not beats_p10

self_check = pd.DataFrame(
    {
        "question": [
            "Does the model beat baseline Precision@10?",
            "Does the model beat baseline Precision@50?",
            "Is the added complexity clearly worth it?",
            "Were forbidden features excluded?",
            "Were all rows scored out-of-fold?",
        ],
        "answer": [
            "yes" if beats_p10 else "no",
            "yes" if beats_p50 else "no",
            "mixed - worth it only for top-50 batch review" if worth_complexity else "no clear win",
            "yes",
            "yes",
        ],
    }
)
display(self_check)
print("Notebook self-check passed. Ready for human review and commit.")


,question,answer
0,Does the model beat baseline Precision@10?,no
1,Does the model beat baseline Precision@50?,yes
2,Is the added complexity clearly worth it?,mixed - worth it only for top-50 batch review
3,Were forbidden features excluded?,yes
4,Were all rows scored out-of-fold?,yes


Notebook self-check passed. Ready for human review and commit.
